In [1]:
import os
import json
from PIL import Image

# create mapping from cityscapes city names to integers

city_names = set()

for fname in os.listdir("/path/to/datasets/Cityscapes/val/images"):
    city_name = fname.split("_")[0]
    city_names.add(city_name)

for fname in os.listdir("/path/to/datasets/Cityscapes/train/images"):
    city_name = fname.split("_")[0]
    city_names.add(city_name)

city_name_to_id = {name: idx+1 for idx, name in enumerate(sorted(city_names))} # city names sorted alphabetically, mapped to integers starting from 1

# mapping is used to create integer ids for filenames : city_000012_000001.png -> id(city)000012000001, where city is mapped to an integer id

In [2]:

def yolo_to_coco(img_dir, label_dir, class_names):
    images = []
    annotations = []
    categories = []

    # COCO categories
    for i, name in enumerate(class_names):
        categories.append({
            "id": i,
            "name": name,
            "supercategory": "none"
        })

    ann_id = 1

    img_files = [f for f in os.listdir(img_dir)
                 if f.endswith((".jpg", ".png", ".jpeg"))]

    for img_file in img_files:
        img_path = os.path.join(img_dir, img_file)

        if not "Cityscapes" in img_dir:  # Cityscapes has non-numeric filenames
            img_id = int(os.path.splitext(img_file)[0])  # assumes filename is numeric, which is not the case for Cityscapes, but for KITTI and PascalVOC
        else:
            city = img_file.split("_")[0]
            city_id = city_name_to_id[city]

            numeric_img_file = img_file.replace(city, str(city_id))  # remove city name from filename
            img_id = int(numeric_img_file.replace("_", "").split(".")[0])  # remove underscores and .png and convert to int

        label_path = os.path.join(
            label_dir,
            os.path.splitext(img_file)[0] + ".txt"
        )

        # read image size
        with Image.open(img_path) as img:
            w, h = img.size

        images.append({
            "id": img_id,
            "file_name": img_file,
            "width": w,
            "height": h
        })

        # read YOLO labels
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                lines = f.readlines()

            for line in lines:
                cls, xc, yc, bw, bh = map(float, line.strip().split())

                cls = int(cls)

                # YOLO → pixel xyxy
                x_center = xc * w
                y_center = yc * h
                box_w = bw * w
                box_h = bh * h

                x1 = x_center - box_w / 2
                y1 = y_center - box_h / 2

                annotations.append({
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": cls,
                    "bbox": [x1, y1, box_w, box_h],
                    "area": box_w * box_h,
                    "iscrowd": 0
                })

                ann_id += 1

    return {
        "images": images,
        "annotations": annotations,
        "categories": categories
    }

In [3]:
# Pascal VOC -> COCO format from YOLO Labels

label_dir = "/path/to/datasets/VOC/labels/val2012"
image_dir = "/path/to/datasets/VOC/images/val2012"

cat_list=[
        "aeroplane","bicycle","bird","boat","bottle",
        "bus","car","cat","chair","cow",
        "diningtable","dog","horse","motorbike","person",
        "pottedplant","sheep","sofa","train","tvmonitor"
    ]

coco_dict = yolo_to_coco(
            img_dir=image_dir,
            label_dir=label_dir,
            class_names=cat_list
        )

print(len(coco_dict["images"]), len(coco_dict["annotations"]), len(coco_dict["categories"]))

print(coco_dict["images"][0])

with open( os.path.dirname(label_dir) + "/instances_val2012.json", "w") as f:
    json.dump(coco_dict, f)

5823 13841 20
{'id': 2008000002, 'file_name': '2008_000002.jpg', 'width': 500, 'height': 375}


In [4]:
# KITTI -> COCO format from YOLO Labels

label_dir = "/path/to/datasets/KITTI/val/labels"
image_dir = "/path/to/datasets/KITTI/val/images"

cat_list=[ 
            "car", "van", "truck", "pedestrian", "person_sitting", "cyclist", "tram"
        ]

coco_dict = yolo_to_coco(
            img_dir=image_dir,
            label_dir=label_dir,
            class_names=cat_list
        )

print(len(coco_dict["images"]), len(coco_dict["annotations"]), len(coco_dict["categories"]))

print(coco_dict["images"][0])

with open( os.path.dirname(label_dir) + "/instances_val.json", "w") as f:
    json.dump(coco_dict, f)

1497 8008 7
{'id': 2, 'file_name': '000002.png', 'width': 1242, 'height': 375}


In [5]:
# Cityscapes -> COCO format from YOLO Labels

label_dir = "/path/to/datasets/Cityscapes/val/labels"
image_dir = "/path/to/datasets/Cityscapes/val/images"

cat_list=[ 
        "person", "rider", "car", "truck", "bus", "train", "motorcycle", "bicycle"
    ]

coco_dict = yolo_to_coco(
            img_dir=image_dir,
            label_dir=label_dir,
            class_names=cat_list
        )

print(len(coco_dict["images"]), len(coco_dict["annotations"]), len(coco_dict["categories"]))

print(coco_dict["images"][0])

with open( os.path.dirname(label_dir) + "/instances_val.json", "w") as f:
    json.dump(coco_dict, f)

500 10180 8
{'id': 8000000000294, 'file_name': 'frankfurt_000000_000294.png', 'width': 2048, 'height': 1024}
